### #1

In [48]:
import numpy as np
import pandas as pd

#1
np.random.seed(2024)

date_index = pd.date_range(start='2024-03-01', periods=90)

random_values = np.round(np.random.normal(loc=18, scale=4, size=90), 1)

s = pd.Series(random_values, index=date_index)

print(s.head())
print(s.describe())
print('Index:', s.index)
print('Dtype:', s.dtype)

missing_indices = np.random.choice(s.index, size=10, replace=False)
s[missing_indices] = np.nan

valid_indices = s.dropna().index
outlier_indices = np.random.choice(valid_indices, size=5, replace=False)
s[outlier_indices] *= 3

2024-03-01    24.7
2024-03-02    20.9
2024-03-03    17.2
2024-03-04    17.4
2024-03-05    21.7
Freq: D, dtype: float64
count    90.000000
mean     18.202222
std       3.923996
min       7.500000
25%      15.800000
50%      18.300000
75%      21.350000
max      25.700000
dtype: float64
Index: DatetimeIndex(['2024-03-01', '2024-03-02', '2024-03-03', '2024-03-04',
               '2024-03-05', '2024-03-06', '2024-03-07', '2024-03-08',
               '2024-03-09', '2024-03-10', '2024-03-11', '2024-03-12',
               '2024-03-13', '2024-03-14', '2024-03-15', '2024-03-16',
               '2024-03-17', '2024-03-18', '2024-03-19', '2024-03-20',
               '2024-03-21', '2024-03-22', '2024-03-23', '2024-03-24',
               '2024-03-25', '2024-03-26', '2024-03-27', '2024-03-28',
               '2024-03-29', '2024-03-30', '2024-03-31', '2024-04-01',
               '2024-04-02', '2024-04-03', '2024-04-04', '2024-04-05',
               '2024-04-06', '2024-04-07', '2024-04-08', '2024-04-09

In [ ]:
#2
print('결측 발생 날짜 목록:')
print(s[s.isnull()].index)
print(f'결측 개수: {s.isnull().sum()}개')

s_drop = s.dropna()                               
s_zero = s.fillna(0)                              
s_mean = s.fillna(s.mean())                       
s_ffill = s.ffill()                               

comparison_df = pd.DataFrame({
    '결측 제거': [s_drop.mean(), s_drop.std()],
    '0으로 채움': [s_zero.mean(), s_zero.std()],
    '전체 평균 채움': [s_mean.mean(), s_mean.std()],
    '앞값 채움': [s_ffill.mean(), s_ffill.std()]
}, index=['평균', '표준편차'])

print(comparison_df.round(2))

# 시계열 데이터에서 가장 부적절한 방식에 대한 근거
# 시계열 데이터 관점에서 '0으로 채움' 이나 '전체 평균으로 채움' 방식이 가장 부적절합니다.
# 이유는 기온과 같은 시계열 데이터는 연속성과 인접한 날짜 간의 강한 상관관계를 가집니다. 
# 0이나 전체 평균으로 갑자기 채우게 되면 데이터의 흐름이 크게 깨져 왜곡이 발생하고, 평균, 표준편차 등을 크게 훼손하게 됩니다. 반면에 앞값 채움이나 보간법이 시계열 특성 유지에 더 유리합니다.

결측 발생 날짜 목록:
DatetimeIndex(['2024-04-02', '2024-04-06', '2024-04-12', '2024-04-15',
               '2024-04-17', '2024-04-27', '2024-04-28', '2024-05-01',
               '2024-05-02', '2024-05-12'],
              dtype='datetime64[us]', freq=None)
결측 개수: 10개
      결측 제거  0으로 채움  전체 평균 채움  앞값 채움
평균    20.29   18.04     20.29  20.45
표준편차   8.94   10.58      8.42   8.49


In [ ]:
#3
data = s_ffill.copy()

mean_val = data.mean()
std_val = data.std()
upper_bound = mean_val + (2 * std_val)
lower_bound = mean_val - (2 * std_val)

print(f'상한선: {upper_bound:.2f}, 하한선: {lower_bound:.2f}')

outliers_detected = data[(data > upper_bound) | (data < lower_bound)]
print('\n탐지된 이상치 목록:')
print(outliers_detected)
print(f'탐지된 개수: {len(outliers_detected)}개')

cleaned_values = np.where(data > upper_bound, upper_bound, 
                          np.where(data < lower_bound, lower_bound, data))
clean = pd.Series(cleaned_values, index=data.index)

print('보정 전')
print(data.describe())
print('\n보정 후')
print(clean.describe())
# 이상치를 상/하한값으로 대체함에 따라 데이터의 극단적인 값이 제거되어, 보정 후에 표준편차가 감소하고 평균값도 안정적으로 조정되는 것을 확인할 수 있습니다.

상한선: 37.43, 하한선: 3.47

탐지된 이상치 목록:
2024-04-10    65.7
2024-04-13    49.2
2024-04-29    39.9
2024-05-08    47.4
2024-05-18    48.0
dtype: float64
탐지된 개수: 5개
보정 전
count    90.000000
mean     20.451111
std       8.489336
min       7.500000
25%      16.500000
50%      19.550000
75%      22.000000
max      65.700000
dtype: float64

보정 후
count    90.000000
mean     19.750543
std       5.876946
min       7.500000
25%      16.500000
50%      19.550000
75%      22.000000
max      37.429782
dtype: float64


In [ ]:
#4
monthly_summary = clean.groupby(clean.index.month).agg(['count', 'mean', 'min', 'max'])
monthly_summary.index.name = '월'
print('월별 집계 결과:')
print(monthly_summary)

rolling_7d = clean.rolling(window=7).mean()
print('\n7일 이동평균 (상위 10개 행 확인):')
print(rolling_7d.head(10))

print('\n기온 상위 5일:')
print(clean.sort_values(ascending=False).head(5))

overall_mean = clean.mean()
higher_than_mean_count = (clean > overall_mean).sum()
print(f'\n전체 평균({overall_mean:.2f})보다 기온이 높았던 날의 총 일수: {higher_than_mean_count}일')

# 7일 이동평균을 구할 때 앞쪽 6개의 값이 결측으로 나오는 이유는, 윈도우 크기를 채우기에 데이터 개수가 부족하기 때문입니다. 
# 파이썬 롤링 연산은 지정된 창 크기만큼 데이터가 쌓이기 전까지는 계산을 수행할 수 없어 결측 처리됩니다.

월별 집계 결과:
   count       mean   min        max
월                                   
3     31  18.132258   7.5  24.700000
4     30  20.966312  12.7  37.429782
5     29  20.222744   8.5  37.429782

7일 이동평균 (상위 10개 행 확인):
2024-03-01          NaN
2024-03-02          NaN
2024-03-03          NaN
2024-03-04          NaN
2024-03-05          NaN
2024-03-06          NaN
2024-03-07    18.857143
2024-03-08    17.142857
2024-03-09    16.985714
2024-03-10    17.157143
Freq: D, dtype: float64

기온 상위 5일:
2024-04-13    37.429782
2024-04-10    37.429782
2024-04-29    37.429782
2024-05-08    37.429782
2024-05-18    37.429782
dtype: float64

전체 평균(19.75)보다 기온이 높았던 날의 총 일수: 45일


### #2

In [ ]:
#1
np.random.seed(7)
n = 300

order_id = ['ORD' + str(i).zfill(4) for i in range(1, n + 1)]

order_date = [pd.to_datetime('2024-01-01') + pd.DateOffset(days=int(d)) for d in np.random.randint(0, 180, n)]

grade = np.random.choice(['일반', '실버', '골드', 'VIP'], n, p=[0.5, 0.25, 0.15, 0.10])
category = np.random.choice(['식품', '의류', '가전', '도서', '뷰티'], n)
qty = np.random.randint(1, 6, n)
price = np.random.choice([5000, 12000, 25000, 48000, 99000], n)
coupon = np.random.choice([True, False], n, p=[0.3, 0.7])

df = pd.DataFrame({
    '주문번호': order_id, '주문일': order_date, '회원등급': grade,
    '카테고리': category, '수량': qty, '단가': price, '쿠폰사용': coupon
})
df.set_index('주문번호', inplace=True)

print(df.head())
print(df.info())
print(df.describe())

print('\n회원등급별 건수:')
print(df['회원등급'].value_counts())

df['수량'] = df['수량'].astype(float)
missing_indices = np.random.choice(df.index, size=20, replace=False)
df.loc[missing_indices, '수량'] = np.nan

print(f'\n보정 전 결측 개수: {df['수량'].isnull().sum()}개')
df['수량'] = df['수량'].fillna(df['수량'].median())
print(f'보정 후 결측 개수: {df['수량'].isnull().sum()}개')

               주문일 회원등급 카테고리  수량     단가   쿠폰사용
주문번호                                          
ORD0001 2024-06-24   일반   도서   4  12000   True
ORD0002 2024-01-26   일반   식품   3   5000  False
ORD0003 2024-03-08   일반   식품   1  12000   True
ORD0004 2024-05-31   실버   의류   1  12000   True
ORD0005 2024-04-13   일반   가전   4  12000  False
<class 'pandas.DataFrame'>
Index: 300 entries, ORD0001 to ORD0300
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   주문일     300 non-null    datetime64[us]
 1   회원등급    300 non-null    str           
 2   카테고리    300 non-null    str           
 3   수량      300 non-null    int32         
 4   단가      300 non-null    int64         
 5   쿠폰사용    300 non-null    bool          
dtypes: bool(1), datetime64[us](1), int32(1), int64(1), str(2)
memory usage: 13.2+ KB
None
                       주문일          수량            단가
count                  300  300.000000    300.000000
mean   2024-03-31 09:45:

In [ ]:
#2
df['주문금액'] = df['수량'] * df['단가']

grade_discount = {'일반': 0, '실버': 3, '골드': 5, 'VIP': 10}
df['등급할인율'] = df['회원등급'].map(grade_discount)

df['쿠폰할인율'] = np.where(df['쿠폰사용'], 5, 0)
df['총할인율'] = df['등급할인율'] + df['쿠폰할인율']
df['할인금액'] = (df['주문금액'] * df['총할인율'] / 100).round()

df['배송비'] = np.where(df['주문금액'] - df['할인금액'] >= 30000, 0, 3000)
df['최종결제금액'] = df['주문금액'] - df['할인금액'] + df['배송비']

def check_size(val):
    if val >= 200000:
        return '대형'
    elif val >= 50000:
        return '중형'
    else:
        return '소형'

df['주문규모'] = df['최종결제금액'].apply(check_size)
print(df.head())

               주문일 회원등급 카테고리   수량     단가   쿠폰사용  월     주문금액  등급할인율  쿠폰할인율  \
주문번호                                                                        
ORD0001 2024-06-24   일반   도서  4.0  12000   True  6  48000.0      0      5   
ORD0002 2024-01-26   일반   식품  3.0   5000  False  1  15000.0      0      0   
ORD0003 2024-03-08   일반   식품  1.0  12000   True  3  12000.0      0      5   
ORD0004 2024-05-31   실버   의류  1.0  12000   True  5  12000.0      3      5   
ORD0005 2024-04-13   일반   가전  4.0  12000  False  4  48000.0      0      0   

         총할인율    할인금액   배송비   최종결제금액 주문규모  
주문번호                                       
ORD0001     5  2400.0     0  45600.0   소형  
ORD0002     0     0.0  3000  18000.0   소형  
ORD0003     5   600.0  3000  14400.0   소형  
ORD0004     8   960.0  3000  14040.0   소형  
ORD0005     0     0.0     0  48000.0   소형  


In [ ]:
#3
df['월'] = df['주문일'].dt.month

print('\n월별 집계:')
print(df.groupby('월')['최종결제금액'].agg(['count', 'sum', 'mean']).round(0))

print('\n카테고리별, 회원등급별 매출 합계:')
print(df.groupby(['카테고리', '회원등급'])['최종결제금액'].sum())

print('\nVIP이면서 최종결제금액 10만원 이상인 주문 건수:')
vip_high = df[(df['회원등급'] == 'VIP') & (df['최종결제금액'] >= 100000)]
print(f"{len(vip_high)}건")

print('\n쿠폰을 사용한 가전 카테고리 주문 조회:')
print(df[(df['쿠폰사용'] == True) & (df['카테고리'] == '가전')])

print('\n카테고리별 매출 상위 2건:')
top_2 = df.sort_values(by=['카테고리', '최종결제금액'], ascending=[True, False]).groupby('카테고리').head(2)
print(top_2[['카테고리', '최종결제금액']])


월별 집계:
   count        sum      mean
월                            
1     46  4643650.0  100949.0
2     54  4789570.0   88696.0
3     48  5317610.0  110784.0
4     50  5501470.0  110029.0
5     56  4126800.0   73693.0
6     46  4633590.0  100730.0

카테고리별, 회원등급별 매출 합계:
카테고리  회원등급
가전    VIP      530350.0
      골드       626500.0
      실버      1582990.0
      일반      3299600.0
도서    VIP      536400.0
      골드       577150.0
      실버      1237160.0
      일반      3124400.0
뷰티    VIP      464250.0
      골드       516750.0
      실버       653610.0
      일반      2599350.0
식품    VIP      609600.0
      골드      1283550.0
      실버      2179450.0
      일반      3893050.0
의류    VIP      839500.0
      골드       682550.0
      실버      1463930.0
      일반      2312550.0
Name: 최종결제금액, dtype: float64

VIP이면서 최종결제금액 10만원 이상인 주문 건수:
12건

쿠폰을 사용한 가전 카테고리 주문 조회:
               주문일 회원등급 카테고리   수량     단가  쿠폰사용  월      주문금액  등급할인율  쿠폰할인율  \
주문번호                                                                       

In [ ]:
#4
df.drop(['등급할인율', '쿠폰할인율'], axis='columns', inplace=True)

df.to_csv('orders.csv', encoding='utf-8-sig')

df_read = pd.read_csv('orders.csv', index_col='주문번호', header=0, sep=',', parse_dates=['주문일'])

print('\nloc 결과')
print(df_read.loc['ORD0010'])

print('\niloc 결과')
print(df_read.iloc[9])

# 두 값이 완벽하게 동일한 이유는 데이터를 처음 생성할 때 인덱스(주문번호)를 'ORD0001'부터 차례대로 부여했기 때문입니다. 
# loc은 명시적 인덱스 이름을 찾아가고, iloc은 묵시적 순서를 찾아가게 되는데, 행이 중간에 삭제되거나 섞이지 않았으므로 두 방식이 동일한 행을 가리키게 됩니다.


loc 결과
주문일       2024-03-30 00:00:00
회원등급                       실버
카테고리                       도서
수량                        4.0
단가                      48000
쿠폰사용                    False
월                           3
주문금액                 192000.0
총할인율                        3
할인금액                   5760.0
배송비                         0
최종결제금액               186240.0
주문규모                       중형
Name: ORD0010, dtype: object

iloc 결과
주문일       2024-03-30 00:00:00
회원등급                       실버
카테고리                       도서
수량                        4.0
단가                      48000
쿠폰사용                    False
월                           3
주문금액                 192000.0
총할인율                        3
할인금액                   5760.0
배송비                         0
최종결제금액               186240.0
주문규모                       중형
Name: ORD0010, dtype: object


### #3

In [54]:
#1
np.random.seed(99)

branches = ['서울', '부산', '대구', '광주', '대전']
products = ['A', 'B', 'C']
quarters = ['1Q', '2Q', '3Q', '4Q']

def make_quarter(q):
    idx = pd.MultiIndex.from_product([branches, products], names=['지점', '상품'])
    data = {
        '판매량': np.random.randint(50, 300, len(idx)),
        '단가':   np.random.choice([10000, 15000, 20000], len(idx)),
        '반품':   np.random.randint(0, 20, len(idx)),
    }
    df = pd.DataFrame(data, index=idx)
    df['분기'] = q
    return df

sales = pd.concat([make_quarter(q) for q in quarters])

print('Shape:', sales.shape)
print('Index Names:', sales.index.names)
print(sales.head())

sales['판매량'] = sales['판매량'].astype(float)
drop_idx = np.random.choice(len(sales), 8, replace=False)
sales.iloc[drop_idx, sales.columns.get_loc('판매량')] = np.nan
print('결측 개수:', sales['판매량'].isnull().sum())

med = sales.groupby('지점')['판매량'].transform('median')
sales['판매량'] = sales['판매량'].fillna(med)
print('보정 후 결측 개수:', sales['판매량'].isnull().sum())

Shape: (60, 4)
Index Names: ['지점', '상품']
       판매량     단가  반품  분기
지점 상품                    
서울 A   179  15000  14  1Q
   B    85  10000   4  1Q
   C   235  15000  12  1Q
부산 A   218  20000  17  1Q
   B   251  10000   9  1Q
결측 개수: 8
보정 후 결측 개수: 0


In [55]:
#2
sales['매출'] = sales['판매량'] * sales['단가']
sales['반품율'] = (sales['반품'] / sales['판매량'] * 100).round(2)

sales['등급'] = np.where(sales['매출'] >= 4000000, 'A',
                  np.where(sales['매출'] >= 2000000, 'B', 'C'))

print('\n지점 기준 매출 집계:')
print(sales.groupby('지점')['매출'].agg(['count', 'sum', 'mean', 'max']).round(0))

print('\n지점, 상품 기준 매출 합계:')
print(sales.groupby(['지점', '상품'])['매출'].sum())

print('\n분기, 지점 기준 매출 합계:')
print(sales.groupby(['분기', '지점'])['매출'].sum())

print('\n등급별 행 개수:')
print(sales.groupby('등급').size())


지점 기준 매출 집계:
    count         sum       mean        max
지점                                         
광주     12  28530000.0  2377500.0  4220000.0
대구     12  32235000.0  2686250.0  4740000.0
대전     12  37295000.0  3107917.0  5540000.0
부산     12  28435000.0  2369583.0  5640000.0
서울     12  31195000.0  2599583.0  4300000.0

지점, 상품 기준 매출 합계:
지점  상품
광주  A     11690000.0
    B      6750000.0
    C     10090000.0
대구  A     10110000.0
    B     11520000.0
    C     10605000.0
대전  A     11205000.0
    B     15245000.0
    C     10845000.0
부산  A      7925000.0
    B      8920000.0
    C     11590000.0
서울  A      8615000.0
    B      9095000.0
    C     13485000.0
Name: 매출, dtype: float64

분기, 지점 기준 매출 합계:
분기  지점
1Q  광주     6880000.0
    대구     6250000.0
    대전     9605000.0
    부산    12510000.0
    서울     7060000.0
2Q  광주     6420000.0
    대구     5995000.0
    대전    13380000.0
    부산     5075000.0
    서울     6795000.0
3Q  광주     6780000.0
    대구    11215000.0
    대전     7830000.0
    부산     4440

In [57]:
#3
total = sales.groupby('지점')['매출'].sum().sort_values(ascending=False)
print('\n지점별 연간 매출 합계 및 순위:')
print(total)
print(total.rank(ascending=False))

pv = sales.pivot_table(index='지점', columns='분기', values='매출', aggfunc='sum')
pv['연간합계'] = pv.sum(axis=1)
pv = pv.sort_values(by='연간합계', ascending=False)
print('\n교차표:')
print(pv)

pv['성장률'] = ((pv['4Q'] - pv['1Q']) / pv['1Q'] * 100).round(1)
print('\n성장률 기준 정렬:')
print(pv[['1Q', '4Q', '성장률']].sort_values(by='성장률', ascending=False))


지점별 연간 매출 합계 및 순위:
지점
대전    37295000.0
대구    32235000.0
서울    31195000.0
광주    28530000.0
부산    28435000.0
Name: 매출, dtype: float64
지점
대전    1.0
대구    2.0
서울    3.0
광주    4.0
부산    5.0
Name: 매출, dtype: float64

교차표:
분기          1Q          2Q          3Q         4Q        연간합계
지점                                                           
대전   9605000.0  13380000.0   7830000.0  6480000.0  37295000.0
대구   6250000.0   5995000.0  11215000.0  8775000.0  32235000.0
서울   7060000.0   6795000.0   9910000.0  7430000.0  31195000.0
광주   6880000.0   6420000.0   6780000.0  8450000.0  28530000.0
부산  12510000.0   5075000.0   4440000.0  6410000.0  28435000.0

성장률 기준 정렬:
분기          1Q         4Q   성장률
지점                             
대구   6250000.0  8775000.0  40.4
광주   6880000.0  8450000.0  22.8
서울   7060000.0  7430000.0   5.2
대전   9605000.0  6480000.0 -32.5
부산  12510000.0  6410000.0 -48.8


In [ ]:
#4
a = sales[sales['분기'] == '1Q'].groupby('지점')['매출'].sum()
b = sales[sales['분기'] == '4Q'].groupby('지점')['매출'].sum()
side = pd.concat([a, b], axis=1, keys=['1Q', '4Q'])
print('\n1Q, 4Q 비교표:')
print(side)

top = sales.sort_values(by='매출', ascending=False).groupby('지점').head(1)
print('\n지점별 매출 1위 주문:')
print(top[['분기', '매출', '등급']])

sales.to_csv('sales_report.csv', encoding='utf-8-sig')
r = pd.read_csv('sales_report.csv', index_col=['지점', '상품'], header=0)

print('\nloc["서울"] 조회 결과: ')
print(r.loc['서울'])

# loc으로 첫 번째 레벨인 서울을 지정하여 조회하면, 멀티 인덱스의 첫 번째 레벨이 소모되면서 사라지게 됩니다. 
# 그 결과 원본 sales는 지점과 상품을 동시에 인덱스로 갖는 2중 MultiIndex 구조였지만, r.loc['서울']을 수행한 후의 결과는 상품명만 인덱스로 남는 단일 인덱스 구조로 달라지게 됩니다.


1Q, 4Q 비교표:
            1Q         4Q
지점                       
광주   6880000.0  8450000.0
대구   6250000.0  8775000.0
대전   9605000.0  6480000.0
부산  12510000.0  6410000.0
서울   7060000.0  7430000.0

지점별 매출 1위 주문:
       분기         매출 등급
지점 상품                  
부산 C   1Q  5640000.0  A
대전 B   2Q  5540000.0  A
대구 B   3Q  4740000.0  A
서울 B   3Q  4300000.0  A
광주 A   2Q  4220000.0  A

loc["서울"] 조회 결과: 
      판매량     단가  반품  분기         매출    반품율 등급
상품                                           
A   179.0  15000  14  1Q  2685000.0   7.82  B
B    85.0  10000   4  1Q   850000.0   4.71  C
C   235.0  15000  12  1Q  3525000.0   5.11  B
A   164.0  10000   8  2Q  1640000.0   4.88  C
B   125.0  15000   3  2Q  1875000.0   2.40  C
C   164.0  20000  13  2Q  3280000.0   7.93  B
A   141.0  10000  16  3Q  1410000.0  11.35  C
B   215.0  20000  18  3Q  4300000.0   8.37  A
C   210.0  20000  10  3Q  4200000.0   4.76  A
A   144.0  20000   2  4Q  2880000.0   1.39  B
B   207.0  10000   7  4Q  2070000.0   3.38  B
C   1